# Análise Completa de Ações Brasileiras com IA

Notebook pronto para rodar localmente que baixa dados históricos da B3 via Yahoo Finance (yfinance), oferece alternativa offline via arquivos COTAHIST, calcula indicadores técnicos, compara múltiplos tickers e ilustra uma previsão simples com ARIMA. Os gráficos usam Plotly para interatividade dentro do Jupyter.

## Dependências

Execute a célula abaixo uma única vez no ambiente local para instalar os pacotes necessários (ou adapte para um `requirements.txt`).

In [ ]:
# Remova o comentário para instalar dependências no ambiente local
# %pip install pandas numpy yfinance plotly statsmodels requests

In [ ]:
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import yfinance as yf
from statsmodels.tsa.arima.model import ARIMA

pd.options.display.float_format = "{:.2f}".format

## Utilitários de entrada

- `normalizar_ticker`: adiciona o sufixo `.SA` quando ausente para B3.
- `selecionar_tickers`: aceita lista manual ou busca tickers por setor via BRAPI (quando houver conexão). Em cenários offline, usa um mapa local mínimo para alguns setores populares.

In [ ]:
BRAPI_LIST_URL = "https://brapi.dev/api/quote/list"

LOCAL_SECTORS = {
    "energia": ["PETR4", "PRIO3", "RECV3"],
    "mineracao": ["VALE3", "CSNA3", "GGBR4"],
    "financeiro": ["ITUB4", "BBDC4", "BBAS3"],
    "varejo": ["LREN3", "MGLU3", "AMER3"],
}

def normalizar_ticker(ticker: str) -> str:
    ticker = ticker.strip().upper()
    return ticker if ticker.endswith(".SA") else f"{ticker}.SA"

def selecionar_tickers(tickers=None, setor=None, limite: int = 5):
    if tickers:
        return [normalizar_ticker(t) for t in tickers]
    if not setor:
        raise ValueError("Informe tickers manualmente ou um setor para busca.")

    setor_lower = setor.lower()
    params = {"search": "", "limit": limite, "sector": setor_lower}
    try:
        response = requests.get(BRAPI_LIST_URL, params=params, timeout=10)
        response.raise_for_status()
        data = response.json().get("stocks", [])
        if data:
            return [normalizar_ticker(item["stock"] + ".SA") for item in data[:limite]]
    except Exception:
        pass

    locais = LOCAL_SECTORS.get(setor_lower)
    if locais:
        return [normalizar_ticker(t) for t in locais[:limite]]
    raise ValueError(f"Nenhum ticker encontrado para o setor '{setor}'.")

## Coleta de dados de preços

- `baixar_historico_yf`: usa yfinance para 2 anos de dados (ajuste automático para o sufixo `.SA`).
- `baixar_cotahist_b3`: download opcional dos arquivos oficiais COTAHIST e parsing para DataFrame (útil para uso totalmente offline). Informe anos como `[2023, 2024]`, por exemplo.

In [ ]:
COTAHIST_URL = "https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_AAAA.ZIP"

def baixar_historico_yf(tickers, periodo: str = "2y"):
    historicos = {}
    for t in tickers:
        ticker_fmt = normalizar_ticker(t)
        hist = yf.Ticker(ticker_fmt).history(period=periodo, auto_adjust=False)
        if hist.empty:
            continue
        hist = hist.rename(columns=str.capitalize).reset_index().rename(columns={"Date": "Data"})
        historicos[ticker_fmt] = hist
    return historicos

def _parse_cotahist_file(conteudo: bytes):
    linhas = conteudo.decode("latin-1").splitlines()
    registros = []
    for linha in linhas:
        if not linha.startswith("00") and len(linha) >= 245:
            data = pd.to_datetime(linha[2:10], format="%Y%m%d")
            ticker = linha[12:24].strip()
            preco_abertura = float(linha[56:69]) / 100
            preco_maximo = float(linha[69:82]) / 100
            preco_minimo = float(linha[82:95]) / 100
            preco_medio = float(linha[95:108]) / 100
            preco_fechamento = float(linha[108:121]) / 100
            volume = int(linha[170:188])
            registros.append({
                "Data": data,
                "Ticker": ticker,
                "Open": preco_abertura,
                "High": preco_maximo,
                "Low": preco_minimo,
                "Close": preco_fechamento,
                "Average": preco_medio,
                "Volume": volume,
            })
    return pd.DataFrame(registros)

def baixar_cotahist_b3(anos):
    frames = []
    for ano in anos:
        url = COTAHIST_URL.replace("AAAA", str(ano))
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        frames.append(_parse_cotahist_file(resp.content))
    return pd.concat(frames, ignore_index=True)

## Indicadores técnicos e métricas auxiliares

- `calcular_indicadores`: cria médias móveis, volatilidade de 30 dias, RSI, retorno diário e acumulado.
- `matriz_correlacao`: correlação de retornos diários entre múltiplos ativos.

In [ ]:
def _calcular_rsi(series: pd.Series, periodo: int = 14):
    delta = series.diff()
    ganhos = delta.clip(lower=0)
    perdas = -delta.clip(upper=0)
    ganho_rol = ganhos.rolling(periodo).mean()
    perda_rol = perdas.rolling(periodo).mean()
    rs = ganho_rol / perda_rol
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calcular_indicadores(df: pd.DataFrame):
    df = df.copy()
    df["Retorno"] = df["Close"].pct_change()
    df["Retorno_Acumulado"] = (1 + df["Retorno"]).cumprod() - 1
    df["MA50"] = df["Close"].rolling(50).mean()
    df["MA200"] = df["Close"].rolling(200).mean()
    df["Volatilidade30"] = df["Retorno"].rolling(30).std() * math.sqrt(252)
    df["RSI14"] = _calcular_rsi(df["Close"], 14)
    return df

def matriz_correlacao(historicos):
    retornos = []
    nomes = []
    for ticker, df in historicos.items():
        nomes.append(ticker)
        retornos.append(df.set_index("Data")["Close"].pct_change())
    combinado = pd.concat(retornos, axis=1)
    combinado.columns = nomes
    return combinado.corr()

## Projeção simples com ARIMA

Função utilitária para estimar um ARIMA(5,1,0) na série de fechamento ajustado e projetar 30 dias. Para análises mais robustas, ajuste a ordem e valide resíduos.

In [ ]:
def prever_arima(series: pd.Series, dias: int = 30):
    serie = series.dropna()
    modelo = ARIMA(serie, order=(5, 1, 0)).fit()
    forecast = modelo.forecast(steps=dias)
    forecast.index = pd.date_range(start=serie.index[-1] + pd.Timedelta(days=1), periods=dias, freq="B")
    return forecast

## Síntese textual de insights

`gerar_insights` cria um resumo simples combinando desempenho recente, volatilidade e posição das médias móveis para sugerir compra, manutenção ou venda.

In [ ]:
def gerar_insights(df: pd.DataFrame, ticker: str):
    ultimo = df.dropna().iloc[-1]
    retorno_12m = df.set_index("Data")["Close"].pct_change(252).iloc[-1]
    volatilidade = df["Volatilidade30"].iloc[-1]
    tendencia_alta = ultimo["Close"] > ultimo.get("MA200", np.nan)
    curto_acima_longo = ultimo.get("MA50", np.nan) > ultimo.get("MA200", np.nan)
    rsi = ultimo.get("RSI14", np.nan)

    recomendacao = "Manter"
    if tendencia_alta and curto_acima_longo and rsi < 70:
        recomendacao = "Comprar"
    elif not tendencia_alta and rsi > 60:
        recomendacao = "Vender"

    linhas = [
        f"**{ticker}**",
        f"Retorno aproximado em 12m: {retorno_12m:,.1%}",
        f"Volatilidade anualizada (30d): {volatilidade:,.1%}",
        f"Preço atual vs MA200: {'acima' if tendencia_alta else 'abaixo'}",
        f"RSI(14): {rsi:0.1f}",
        f"Recomendação sugerida: **{recomendacao}**",
    ]
    return "\n".join(linhas)

## Exemplo de uso

Defina tickers manualmente ou informe um setor (ex.: `"energia"`, `"financeiro"`). Ajuste o período conforme necessário.

In [ ]:
tickers = selecionar_tickers(tickers=["PETR4", "VALE3", "ITUB4"])
periodo = "2y"

historicos = baixar_historico_yf(tickers, periodo)
print(f"Foram carregados {len(historicos)} tickers: {list(historicos)}")

### Visualização e indicadores

Calcula indicadores para cada ativo e gera gráficos interativos.

In [ ]:
historicos_ind = {t: calcular_indicadores(df) for t, df in historicos.items()}

figs = []
for ticker, df in historicos_ind.items():
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["Data"], y=df["Close"], name="Fechamento"))
    fig.add_trace(go.Scatter(x=df["Data"], y=df["MA50"], name="MA50"))
    fig.add_trace(go.Scatter(x=df["Data"], y=df["MA200"], name="MA200"))
    fig.update_layout(title=f"{ticker} - Preço e Médias", xaxis_title="Data", yaxis_title="Preço (BRL)")
    figs.append(fig)
figs

In [ ]:
corr = matriz_correlacao(historicos)
corr.style.background_gradient(cmap="RdBu")

### Previsão de tendência

Executa ARIMA para o primeiro ticker disponível e plota a projeção de 30 dias úteis.

In [ ]:
if historicos_ind:
    primeiro = next(iter(historicos_ind.items()))
    ticker, df = primeiro
    serie = df.set_index("Data")["Close"]
    previsao = prever_arima(serie)

    fig_prev = go.Figure()
    fig_prev.add_trace(go.Scatter(x=serie.index, y=serie, name="Histórico"))
    fig_prev.add_trace(go.Scatter(x=previsao.index, y=previsao, name="ARIMA +30d"))
    fig_prev.update_layout(title=f"{ticker} - Projeção ARIMA", xaxis_title="Data", yaxis_title="Preço (BRL)")
    fig_prev.show()

### Insights e recomendação

Gera um resumo por ativo combinando tendência, volatilidade e RSI.

In [ ]:
for ticker, df in historicos_ind.items():
    print(gerar_insights(df, ticker))
    print("-" * 60)

Gerado em 2025-11-29 16:37 UTC